# Membuat Model FCM FuzzyFlix
Notebook ini membuat model Fuzzy C-Means dari data Ayudia (`data/processed/`) dan menyimpan hasilnya ke `models/`.

Parameter mengikuti PRD: m = 2, ε = 1e-5, maxiter = 300, seed = 42. FCM dijalankan pada PCA 2 komponen dari X, karena pada X penuh dengan m = 2 semua pusat cluster menyatu (FPC = 1/c).

In [9]:
import sys, json, warnings
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.fcm import fcm, select_c, membership

DATA = ROOT / 'data' / 'processed'
MODELS = ROOT / 'models'
M, EPS, MAXITER, SEED = 2.0, 1e-5, 300, 42
CLUSTER_COLORS = ['#4C78A8', '#F58518', '#54A24B', '#E45756',
                  '#72B7B2', '#EECA3B', '#B279A2', '#9D755D']

## 1. Muat data
Kolom `Unnamed: 0` hanya nomor baris lama, jadi dibuang.

In [10]:
X_df = pd.read_csv(DATA / 'X_final.csv').drop(columns=['Unnamed: 0'])
film = pd.read_csv(DATA / 'movies_final.csv')
X = X_df.values
print('X:', X.shape, '| film:', film.shape)

X: (9089, 14) | film: (9089, 26)


## 2. PCA 2 komponen dari X
Ruang ini sama dengan scatter "PCA 2D seluruh X" di PRD.

In [11]:
pca_fcm = PCA(n_components=2, random_state=SEED)
Z = pca_fcm.fit_transform(X)
print('Variansi tersimpan:', round(pca_fcm.explained_variance_ratio_.sum(), 3))

Variansi tersimpan: 0.325


## 3. Pilih jumlah cluster c (3 sampai 8)
Aturan PRD: kombinasi FPC dan silhouette terbaik. Di sini skor = FPC + silhouette.

In [12]:
tabel = select_c(Z, c_range=range(3, 9), m=M, error=EPS, maxiter=MAXITER, seed=SEED)
tabel['skor'] = tabel['fpc'] + tabel['silhouette']
C = int(tabel.loc[tabel['skor'].idxmax(), 'c'])
print(tabel.round(3).to_string(index=False))
print('c terpilih =', C)

 c   fpc  silhouette  skor
 3 0.639       0.391 1.030
 4 0.578       0.380 0.959
 5 0.536       0.349 0.885
 6 0.514       0.373 0.887
 7 0.513       0.376 0.889
 8 0.510       0.405 0.915
c terpilih = 3


## 4. Jalankan FCM final

In [13]:
hasil = fcm(Z, C, m=M, error=EPS, maxiter=MAXITER, seed=SEED)
U, V = hasil['U'], hasil['V']
label = U.argmax(axis=1)
sil = silhouette_score(Z, label)
print(f"Iterasi: {hasil['n_iter']} | FPC: {hasil['fpc']:.3f} | Silhouette: {sil:.3f}")

Iterasi: 61 | FPC: 0.639 | Silhouette: 0.391


## 5. Beri nama cluster
Nama diambil dari genre yang paling sering muncul pada film yang dominan di cluster itu, ditambah rata-rata rating.

In [14]:
genre_list = film['genres'].apply(lambda s: [g['name'] for g in literal_eval(s)])
baris = []
for j in range(C):
    idx = np.where(label == j)[0]
    top = (genre_list.iloc[idx].explode().value_counts() / len(idx)).head(3)  # persen film yang punya genre itu
    nama = ' & '.join(top.index[:2]) if top.iloc[1] >= 0.5 else top.index[0]
    baris.append({
        'cluster': f'C{j + 1}',
        'nama': nama,
        'warna': CLUSTER_COLORS[j],
        'jumlah_film': len(idx),
        'rata_rating': round(film['vote_average'].iloc[idx].mean(), 1),
        'genre_teratas': ', '.join(f'{g} ({p:.0%})' for g, p in top.items()),
    })
deskripsi = pd.DataFrame(baris)
deskripsi

,cluster,nama,warna,jumlah_film,rata_rating,genre_teratas
0,C1,Comedy,#4C78A8,3053,6.2,"Comedy (81%), Family (26%), Romance (20%)"
1,C2,Thriller & Action,#F58518,2667,6.2,"Thriller (66%), Action (52%), Drama (29%)"
2,C3,Drama,#54A24B,3369,6.7,"Drama (90%), Romance (24%), Thriller (21%)"


## 6. Simpan model ke `models/`
`U.npy` dan `V.npy` jangan di-commit (sesuai PRD).

In [15]:
MODELS.mkdir(exist_ok=True)
np.save(MODELS / 'U.npy', U)
np.save(MODELS / 'V.npy', V)
joblib.dump(pca_fcm, MODELS / 'pca_fcm.joblib')
deskripsi.to_csv(MODELS / 'deskripsi_cluster.csv', index=False)
meta = {'m': M, 'c': C, 'k_pca': 2, 'epsilon': EPS, 'maxiter': MAXITER, 'seed': SEED,
        'n_iter': int(hasil['n_iter']), 'fpc': float(hasil['fpc']), 'silhouette': float(sil),
        'J_history': [float(j) for j in hasil['J_history']]}
(MODELS / 'fcm_meta.json').write_text(json.dumps(meta, indent=2))
print('Tersimpan:', sorted(p.name for p in MODELS.iterdir()))

Tersimpan: ['.gitkeep', 'U.npy', 'V.npy', 'deskripsi_cluster.csv', 'fcm_meta.json', 'pca_fcm.joblib']


In [17]:
# Cek hitung manual (spreadsheet docs/hitung_manual_fcm.xlsx) dengan fcm.py
X_manual = np.array([[-0.21, 0.40], [-0.13, 0.38], [-0.50, 0.22],
                     [-0.49, -0.01], [-0.54, -0.04]])
U_awal = np.array([[0.6, 0.4], [0.7, 0.3], [0.4, 0.6], [0.3, 0.7], [0.5, 0.5]])

iter1 = fcm(X_manual, 2, m=2.0, error=1e-4, maxiter=1, init=U_awal)
print('Iterasi 1')
print('V =', iter1['V'].round(4).tolist())
print('U baru =', iter1['U'].round(4).tolist())
print('J =', round(iter1['J_history'][0], 4))

akhir = fcm(X_manual, 2, m=2.0, error=1e-4, maxiter=100, init=U_awal)
print('\nSampai konvergen')
print('Jumlah iterasi =', akhir['n_iter'])
print('V akhir =', akhir['V'].round(4).tolist())
print('U akhir =', akhir['U'].round(3).tolist())
print('FPC =', round(akhir['fpc'], 4))

Iterasi 1
V = [[-0.2951, 0.2626], [-0.4447, 0.1204]]
U baru = [[0.8361, 0.1639], [0.8022, 0.1978], [0.2286, 0.7714], [0.145, 0.855], [0.1867, 0.8133]]
J = 0.1435

Sampai konvergen
Jumlah iterasi = 8
V akhir = [[-0.1772, 0.3863], [-0.5111, 0.0352]]
U akhir = [[0.994, 0.006], [0.991, 0.009], [0.206, 0.794], [0.01, 0.99], [0.02, 0.98]]
FPC = 0.9171


## 7. Cek cepat
Keanggotaan film baru dihitung dengan `membership(pca_fcm.transform(x), V, m)`. Hasilnya harus sama dengan baris U.

In [16]:
u_cek = membership(pca_fcm.transform(X[:1]), V, M)
print('Sama dengan U[0]:', np.allclose(u_cek, U[0]))

Sama dengan U[0]: True
